In [1]:
# %load_ext cudf.pandas
import numpy as np
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
passmark = 40
df = pd.read_csv("/home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/input/StudentsPerformance.csv")
df = pd.concat([df]*factor)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count    Dtype 
---  ------                       --------------    ----- 
 0   gender                       1000000 non-null  object
 1   race/ethnicity               1000000 non-null  object
 2   parental level of education  1000000 non-null  object
 3   lunch                        1000000 non-null  object
 4   test preparation course      1000000 non-null  object
 5   math score                   1000000 non-null  int64 
 6   reading score                1000000 non-null  int64 
 7   writing score                1000000 non-null  int64 
dtypes: int64(3), object(5)
memory usage: 68.7+ MB


In [4]:
%%time
### cell 0 ###
_ = df.isnull().sum()

CPU times: user 18.6 s, sys: 372 ms, total: 19 s
Wall time: 18.8 s


In [5]:
%%time
### cell 1 ###

df['math score'] = df['math score'].astype(float).fillna(-1)  # Replace NaNs with a default value
# Vectorized conditional assignment using cuDF
df['Math_PassStatus'] = (df['math score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's optimized value_counts()
_ = df['Math_PassStatus'].value_counts()

CPU times: user 41.1 s, sys: 3.61 s, total: 44.7 s
Wall time: 44.5 s


In [6]:
### cell 2 ###
df['reading score'] = df['reading score'].astype(float).fillna(-1)  # Replace NaNs if needed
# Use cuDF's vectorized conditional assignment
df['Reading_PassStatus'] = (df['reading score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})
# Use cuDF's value_counts() (faster than pandas on GPU)
_ = df['Reading_PassStatus'].value_counts()


In [7]:
%%time
### cell 3 ###
df['writing score'] = df['writing score'].astype('float32')
# Use cuDF's boolean indexing + direct assignment (avoids unnecessary operations)
df['Writing_PassStatus'] = 'F'  # Default all to 'F' first
df.loc[df['writing score'] >= passmark, 'Writing_PassStatus'] = 'P'  # Assign 'P' where needed
# Efficient value_counts() with cuDF
_ = df['Writing_PassStatus'].value_counts()

CPU times: user 5.78 s, sys: 1.01 s, total: 6.78 s
Wall time: 6.77 s


In [8]:
%%time
### cell 4 ###
df['OverAll_PassStatus'] = ((df['Math_PassStatus'] == 'P') & 
                            (df['Reading_PassStatus'] == 'P') & 
                            (df['Writing_PassStatus'] == 'P')).astype("str").replace({'True': 'P', 'False': 'F'})

# Optimized cuDF value_counts()
_ = df['OverAll_PassStatus'].value_counts()

CPU times: user 54 s, sys: 3.17 s, total: 57.1 s
Wall time: 56.9 s


In [9]:
%%time
### cell 5 ###
df['Total_Marks'] = df['math score']+df['reading score']+df['writing score']
df['Percentage'] = df['Total_Marks']/3

CPU times: user 715 ms, sys: 990 ms, total: 1.71 s
Wall time: 1.7 s


In [ ]:
%%time
### cell 6 ###
df['Grade'] = 'F'

# Apply vectorized conditional assignments
df.loc[df['OverAll_PassStatus'] == 'F', 'Grade'] = 'F'
df.loc[(df['Percentage'] >= 80) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'A'
df.loc[(df['Percentage'] >= 70) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'B'
df.loc[(df['Percentage'] >= 60) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'C'
df.loc[(df['Percentage'] >= 50) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'D'
df.loc[(df['Percentage'] >= 40) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'E'

# Efficient value counts
_ = df['Grade'].value_counts()